In [ ]:
import pandas as pd
import os
import io
from io import BytesIO
from ftplib import FTP
from dotenv import load_dotenv
load_dotenv()

In [26]:
def ConnectFTP(server, username, password):
    """
    Descripcion: Esta funcion establece una conexion FTP con el servidor especificado.
    Parametros:
    server (str): La direccion del servidor FTP.
    username (str): El nombre de usuario para la conexion FTP.
    password (str): La contrasena para la conexion FTP.
    Retorna: Un objeto FTP conectado al servidor.
    """
    ftp = FTP(timeout=60)                
    ftp.connect(server, 21)               
    ftp.login(user=username, passwd=password)

    ftp.set_pasv(True)                    
    ftp.voidcmd("TYPE I")                 

    print(f"Conectado a {server}")
    return ftp

def UploadCsvToFTP(df, path):
    """
    Descripcion: Esta funcion sube un DataFrame de pandas como un archivo CSV a un servidor FTP.
    Parametros:
    df (DataFrame): El DataFrame que se desea subir.
    path (str): La ruta en el servidor FTP donde se guardará el archivo CSV.
    Retorna: None
    """
    csv_buffer = io.BytesIO()
    df.to_csv(csv_buffer, index=False, encoding='utf-8')
    csv_buffer.seek(0)
    ftp = ConnectFTP(os.getenv('ftp_server'),os.getenv('ftp_user'),os.getenv('ftp_password'))
    remote_path = f"/pruebas/csv/{path}"
    ftp.storbinary(f"STOR {remote_path}", csv_buffer)
    ftp.quit()
    print("Archivo subido correctamente:", remote_path)

def ReadCsvFromFTP(remote_file_path):
    '''
    Descripcion: Esta funcion lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
    Parametros:
    ftp (FTP): Un objeto FTP conectado al servidor.
    remote_file_path (str): La ruta del archivo CSV en el servidor FTP.
    Retorna: Un DataFrame de pandas que contiene los datos del archivo CSV.
    '''
    ftp = ConnectFTP(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
    with BytesIO() as bio:
        ftp.retrbinary(f'RETR {remote_file_path}', bio.write)
        bio.seek(0)
        df = pd.read_csv(bio)
    return df

In [27]:
def main():
    try:
        e2021 = ReadCsvFromFTP(r'/pruebas/csv/EntrevistaInicial/EntrevisaInicial(21).csv')
    except Exception as e:
        print('2)Merge_data / Error al leer el archivo 2021 CSV:', e)
        return
    try:
        e2022 = ReadCsvFromFTP(r'/pruebas/csv/EntrevistaInicial/EntrevisaInicial(22).csv')
    except Exception as e:
        print('2)Merge_data / Error al leer el archivo 2022 CSV:', e)
        return
    try:
        e2023 = ReadCsvFromFTP(r'/pruebas/csv/EntrevistaInicial/EntrevisaInicial(23).csv')
    except Exception as e:
        print('2)Merge_data / Error al leer el archivo 2023 CSV:', e)
        return
    try:
        e2024 = ReadCsvFromFTP(r'/pruebas/csv/EntrevistaInicial/EntrevisaInicial(24).csv')
    except Exception as e:
        print('2)Merge_data / Error al leer el archivo 2024 CSV:', e)
        return 
    try:
        e2025 = ReadCsvFromFTP(r'/pruebas/csv/EntrevistaInicial/EntrevisaInicial(25).csv')
    except Exception as e:
        print('2)Merge_data / Error al leer el archivo 2025 CSV:', e)
        return
    try:
        EntrevistaInicial = pd.concat([e2021, e2022, e2023, e2024, e2025], ignore_index=True)
    except Exception as e:
        print("2)Merge_data / Error al concatenar los DataFrames:", e)
        return
    try:
        EntrevistaInicial = EntrevistaInicial[(EntrevistaInicial['CentroCostoId'] > 58) |(EntrevistaInicial['CentroCostoId'].isin([48, 49]))]
        EntrevistaInicial = EntrevistaInicial[EntrevistaInicial['TipoPacienteId'] == 1]

        UploadCsvToFTP(EntrevistaInicial, '/Moddata/EntrevisaInicial.csv')
        return EntrevistaInicial
    except Exception as e:
        print("2)Merge_data / Error al subir el archivo CSV al servidor FTP:", e)
        return
    
if __name__ == "__main__":
    EntrevistaInicial = main()

Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_24700\3622487226.py:49: DtypeWarning: Columns (302,303,304,305,306,307,308,309,311,313,315,330) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_24700\3622487226.py:49: DtypeWarning: Columns (321,323,324,325,326,327,328,329,330,331,332,333,335,336,339) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Conectado a 192.168.62.13


...

Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_24700\3622487226.py:49: DtypeWarning: Columns (332,333,334,335,336,337,338,339,340,341,343,345,346,347,348) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_24700\3622487226.py:49: DtypeWarning: Columns (260,261,262,263,264,265,266,267,268,269,270) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv//Moddata/EntrevisaInicial.csv
